In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "cat_ejec_dbricks")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_susp = spark.table(f"{catalogo}.{esquema_source}.suspensiones")
df_rein = spark.table(f"{catalogo}.{esquema_source}.reinstalaciones")

In [0]:
df_susp = df_susp.filter(upper(trim(col("GRUPO"))).startswith("EFECT"))
df_rein = df_rein.filter(upper(trim(col("RESULTADO"))).startswith("EFECT"))

In [0]:
def clasifica_estrato(codestrato):
    if codestrato is None: return "Comercial"
    codestrato = int(codestrato)
    if codestrato < 3: return "Estrato Bajo"
    elif 3 <= codestrato < 5: return "Estrato Medio"
    else: return "Estrato Alto"   

In [0]:
estrato_udf = F.udf(clasifica_estrato, StringType())
df_susp = df_susp.withColumn("ESTRATO", estrato_udf(col("ESTRATO")))

In [0]:
df_susp = df_susp.dropna(subset=["SUSCRIPTOR", "CICLO", "ESTRATO"])
df_rein = df_rein.dropna(subset=["SUSCRIPTOR", "CICLO", "ESTRATO"])

In [0]:
def normalizar_ciclo(df):

    ciclo_num = F.regexp_extract(F.upper(F.trim(F.col("CICLO").cast("string"))), r"(\d+)", 1).cast("int")
    ciclo_corregido = (F.when(ciclo_num.between(20, 90) & ((ciclo_num % 10) == 0),ciclo_num / 10)
                        .otherwise(ciclo_num).cast("int")
                        )

    return df.withColumn("CICLO", F.when(ciclo_corregido.between(1, 12),F.concat(F.lit("C"),F.lpad(ciclo_corregido.cast("string"), 2, "0"))
                                        ).otherwise(F.lit(None))
                        )

df_susp = normalizar_ciclo(df_susp)
df_rein = normalizar_ciclo(df_rein)

In [0]:
# Join DF de suspensiones y reconexiones

susp = df_susp.withColumn("_id_susp", F.monotonically_increasing_id())

df_resultado = (
    susp.alias("s")
    .join(
        df_rein.alias("r"),
        (F.col("s.suscriptor") == F.col("r.suscriptor")) &
        (F.col("r.fecha_realizada") > F.col("s.fecha_realizada")),
        "left"
    )
    .withColumn(
        "_diferencia_segundos",
        F.col("r.fecha_realizada").cast("long") -
        F.col("s.fecha_realizada").cast("long")
    )
    .withColumn(
        "_orden",
        F.row_number().over(
            Window
            .partitionBy("_id_susp")
            .orderBy(F.col("_diferencia_segundos").asc_nulls_last())
        )
    ).filter(F.col("_orden") == 1)
    .select(
        "s.*",
        *[
            F.col(f"r.{c}").alias(f"rein_{c}")
            for c in df_rein.columns
            if c != "suscriptor"
        ]
    ).drop("_id_susp")
)

# display(df_resultado)

In [0]:
df_resultado = (df_resultado.withColumn("PERIODO", F.date_format(F.col("Fecha_realizada"), "yyyyMM"))
    .withColumn("DIAS_EJEC_SUSP", F.col("fecha_realizada") - F.col("fecha_grabada") )
    .withColumn("DIAS_EJEC_REINS", F.col("rein_fecha_realizada") - F.col("rein_fecha_grabada") )
    .withColumn("DIAS_REINSTALACION", F.col("rein_fecha_realizada") - F.col("fecha_realizada") )
    .withColumn("DIAS_MORA", F.datediff(F.current_date(), F.col("REIN_ULT_FECPAGO")))
)

In [0]:
display(df_resultado)

In [0]:
df_resultado.printSchema()

In [0]:
df_agg_resultado = (df_resultado.groupBy("SUSCRIPTOR","CICLO","ESTRATO","PERIODO")
                    .agg(F.sum("DEUDA"),
                         F.avg("DIAS_EJEC_SUSP").cast("long").alias("DIAS_EJEC_SUSP"),
                         F.avg("DIAS_EJEC_REINS").cast("long").alias("DIAS_EJEC_REINS"),
                         F.avg("DIAS_REINSTALACION").cast("long").alias("DIAS_REINSTALACION"),
                         F.avg("DIAS_MORA").cast("long").alias("DIAS_MORA"),
                         F.count("codigo").alias("VECES").alias("VECES"))
                    ).withColumn("INGESTION_DATE", F.current_timestamp())

In [0]:
# display(df_agg_resultado)
# df_agg_resultado.describe()
df_agg_resultado.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.suspen_reinsta")

In [0]:
%sql
SELECT *
FROM cat_ejec_dbricks.silver.suspen_reinsta LIMIT 100